# Cuisine ingredient networks: geography, language, religion

Reproduces every figure in `cuisine_figures.pdf` from the raw recipe table.

**Inputs (folder `data/`)**
- `all_cuisines_combined.csv` – region, cuisine, recipe, ingredient (the course dataset; the generic "(general)" cuisines are removed, all other cuisines are kept)
- `FiltredIngredientcuisines65.gexf` – the TF-IDF similarity network supplied with the project (compared with our rebuilt graph in section 8)
- `religion_by_cuisine.csv`, `religion_by_country.csv` – Pew 2020 religious composition via Wikipedia *Religions by country* (some rows were inconsistent in the source; see `data_quality_note`)
- `nodes_with_attributes.csv` – cuisine → country, location, continent, official languages, coloniser group (language and colonial coding were done by hand and are not verified)
- `world.geojson` – country outlines for the maps

**Method in one paragraph.** *CuisinesRawCounts*: every cuisine is a node, two cuisines are joined if they share at least X = 1 ingredient, and the weight is the raw count of shared ingredients. *TF-IDF graph*: each cuisine is a document, each ingredient a term; ingredients used by one cuisine or by more than 80 % of cuisines are dropped; cosine similarity of the TF-IDF vectors is the edge weight; edges below 0.3 are removed; communities come from Louvain (best of 60 seeds).

Run all cells in order (about 5 minutes). Figures are written to `figs/`, tables to `tables/`.

## 1. Setup and data

In [1]:
import os,pickle,sys,json,types
for p in ('figs','work','tables'): os.makedirs(p,exist_ok=True)
import pandas as pd,numpy as np
d=pd.read_csv('data/all_cuisines_combined.csv')
d['ingredient']=d.ingredient.astype(str).str.strip().str.lower()
d=d[~d.cuisine.str.contains(r'\(general\)')]
d=d.drop_duplicates(['cuisine','recipe','ingredient'])
CU=sorted(d.cuisine.unique()); ix={c:i for i,c in enumerate(CU)}
ING=sorted(d.ingredient.unique()); jx={g:j for j,g in enumerate(ING)}
nrec=d.groupby('cuisine').recipe.nunique()
TF=np.zeros((len(CU),len(ING)))
for c,g,k in d.groupby(['cuisine','ingredient']).recipe.nunique().reset_index().itertuples(index=False): TF[ix[c],jx[g]]=k
PRES=(TF>0); df_=PRES.sum(0)
base=types.SimpleNamespace(d=d.copy(),CU=CU,ING=ING,nrec=nrec,TF=TF,PRES=PRES,df_=df_,ix=ix,jx=jx)
print(len(CU),'cuisines,',len(ING),'ingredients,',len(d.recipe.unique()),'distinct recipe names')

144 cuisines, 563 ingredients, 1884 distinct recipe names


## 2. Graphs: CuisinesRawCounts and TF-IDF cosine

In [2]:
import networkx as nx
from networkx.algorithms.community import louvain_communities,modularity
from sklearn.metrics import normalized_mutual_info_score as nmi
N=len(CU); frac=df_/N
def tfidf(thr=.8,mind=2):
    keep=(df_>=mind)&(frac<=thr); X=TF[:,keep]; Xk=np.where(X>0,1+np.log(np.maximum(X,1)),0); dfk=(Xk>0).sum(0)
    idf=np.log((1+N)/(1+dfk))+1; Wt=Xk*idf; Wt/=np.linalg.norm(Wt,axis=1,keepdims=True)+1e-12; C=Wt@Wt.T; np.fill_diagonal(C,0); return C,keep
def louv(C,thr):
    G=nx.Graph(); G.add_nodes_from(CU)
    for i in range(N):
        for j in range(i+1,N):
            if C[i,j]>=thr: G.add_edge(CU[i],CU[j],weight=float(C[i,j]))
    runs=[louvain_communities(G,weight='weight',seed=s) for s in range(60)]
    best=sorted(max(runs,key=lambda c:modularity(G,c,weight='weight')),key=len,reverse=True)
    lab=lambda cs:[[k for k,s in enumerate(cs) if c in s][0] for c in CU]; lb=lab(best); stab=np.mean([nmi(lb,lab(r)) for r in runs[:30]])
    return G,best,modularity(G,best,weight='weight'),stab,lb
RAW=(PRES.astype(int)@PRES.T.astype(int)); np.fill_diagonal(RAW,0)
C8,k8=tfidf(.8); C65,k65=tfidf(.65)
G,best,Q,stab,lb=louv(C8,.3)
print('edges',G.number_of_edges(),'Q',round(Q,3),'stab',round(stab,3),[len(s) for s in best])
for i,s in enumerate(best[:8]): print(i,len(s),sorted(s))
print('isolated',[c for c in CU if G.degree(c)==0])
G65,best65,Q65,stab65,lb65=louv(C65,.3); print('0.65:',[len(s) for s in best65][:8],round(Q65,3),'NMI vs main',round(nmi(lb,lb65),3))
sens={}
for t in (.2,.25,.3,.35,.4,.5):
    g,b,q,s,l=louv(C8,t); sens[t]=dict(edges=g.number_of_edges(),isolates=sum(1 for c in CU if g.degree(c)==0),comms=sum(1 for x in b if len(x)>1),Q=q,nmi_vs_03=nmi(lb,l)); print(t,sens[t])
print('raw graph: edges',int((RAW>0).sum()//2),'of',N*(N-1)//2)
pickle.dump(dict(CU=CU,C8=C8,C65=C65,RAW=RAW,best=[sorted(s) for s in best],Q=Q,stab=stab,lb=lb,sens=sens,k8=k8,k65=k65,nmi65=nmi(lb,lb65),best65=[sorted(s) for s in best65],Q65=Q65),open('work/cu.pkl','wb'))

edges 1452 Q 0.346 stab 0.817 [46, 38, 27, 10, 8, 3, 2, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]
0 46 ['Albanian', 'Alsatian', 'Australian', 'Austrian', 'Belgian', 'British', 'Canadian', 'Cornish', 'Czech', 'Danish', 'Dutch', 'English', 'Finnish', 'French', 'Georgian', 'German', 'Gibraltarian', 'Greek', 'Hungarian', 'Irish', 'Italian', 'Italian-American', 'Jamaican', 'Lithuanian', 'Maltese', 'Midwestern U.S.', 'Mongolian', 'Native American', 'New Zealand', 'North American', 'Northern Irish', 'Norwegian', 'Polish', 'Russian', 'Scottish', 'Serbian', 'Slovenian', 'South African', 'Southern U.S.', 'Surinamese', 'Swedish', 'Swiss', 'Trinidadian and Tobagonian', 'Ukrainian', 'United States', 'Welsh']
1 38 ['Afghan', 'Algerian', 'Bengali', 'Chadian', 'Egyptian', 'Ethiopian', 'Gambian', 'Ghanaian', 'Indian', 'Indonesian', 'Iranian', 'Israeli', 'Jewish Yemenite', 'Kenyan', 'Lebanese', 'Libyan', 'Malawian', 'Malaysian', 'Moroccan', 'Nepali', 'Pakistani', 'Pashtun', 'Punjabi', 'Rwandan', 'Senegalese', 'Sind

0.65: [42, 37, 33, 10, 10, 2, 1, 1] 0.415 NMI vs main 0.719


0.2 {'edges': 4151, 'isolates': 0, 'comms': 3, 'Q': 0.17815439222011673, 'nmi_vs_03': 0.5105748735107762}


0.25 {'edges': 2664, 'isolates': 6, 'comms': 6, 'Q': 0.24747740680328958, 'nmi_vs_03': 0.764847489320855}


0.3 {'edges': 1452, 'isolates': 10, 'comms': 7, 'Q': 0.3455068294901242, 'nmi_vs_03': 1.0}


0.35 {'edges': 746, 'isolates': 17, 'comms': 7, 'Q': 0.48260575739918105, 'nmi_vs_03': 0.7084504839921301}


0.4 {'edges': 376, 'isolates': 25, 'comms': 11, 'Q': 0.6465766077013042, 'nmi_vs_03': 0.6453009652847934}


0.5 {'edges': 143, 'isolates': 59, 'comms': 13, 'Q': 0.8060193319126012, 'nmi_vs_03': 0.612205238544047}
raw graph: edges 10166 of 10296


## 3. Plot style and shared objects

In [3]:
import networkx as nx,matplotlib,matplotlib.patheffects as pe
matplotlib.use('Agg'); import matplotlib.pyplot as plt
from scipy.spatial import ConvexHull
from matplotlib.patches import Polygon
from matplotlib.collections import PatchCollection
from matplotlib.lines import Line2D
from sklearn.cluster import KMeans
from sklearn.metrics import normalized_mutual_info_score as nmi
plt.rcParams.update({'font.family':'Inter','font.size':10,'axes.spines.top':False,'axes.spines.right':False,'axes.edgecolor':'#9a9993','axes.labelcolor':'#52514e','xtick.color':'#52514e','ytick.color':'#52514e','axes.titleweight':'bold','axes.titlesize':12,'axes.titlecolor':'#0b0b0b','axes.titlelocation':'left'})
B,O,A,V,Y,GR,INK,MUTE='#2a78d6','#eb6834','#1baf7a','#6250d6','#eda100','#9a9993','#0b0b0b','#52514e'; CC=[B,V,O,A,Y]; halo=[pe.withStroke(linewidth=2.4,foreground='white')]
X=pickle.load(open('work/cu.pkl','rb')); C8,RAW=X['C8'],X['RAW']; best=X['best']; N=len(CU); frac=df_/N
big=[s for s in best if len(s)>=8]; cc={c:i for i,s in enumerate(big) for c in s}
CN=['Northern & central European,\nAnglo & U.S.','Middle Eastern, South Asian\n& East African','Latin American, Iberian\n& U.S. Southwest','West African','East & SE Asian']; CNl=[s.replace('\n',' ') for s in CN]
NA=pd.read_csv('data/nodes_with_attributes.csv',index_col=0)
G=nx.Graph(); G.add_nodes_from(CU)
for i in range(N):
    for j in range(i+1,N):
        if C8[i,j]>=.3: G.add_edge(CU[i],CU[j],weight=C8[i,j])
USG=['United States','Southern U.S.','Texas','Tex-Mex']; US_ALL=USG+['Californian','Louisiana','Midwestern U.S.','Southwestern','Italian-American']
RULE='Rules: every cuisine is a node; (general) cuisines removed, all others kept (144 cuisines).'

## 4. Figures – ingredient filters, TF-IDF network, strongest edges, community maps, unique ingredients (pages 3, 4, 6, 7, 8)

In [4]:
fig,axs=plt.subplots(1,2,figsize=(14,5.6),dpi=200)
ax=axs[0]; bins=np.r_[0.5,1.5,2.5,3.5,5.5,8.5,13.5,21.5,34.5,55.5,89.5,144.5]; h,_=np.histogram(df_,bins); lab=['1','2','3','4–5','6–8','9–13','14–21','22–34','35–55','56–89','90+']
col=[O]+[B]*10; ax.bar(range(len(h)),h,color=col,width=.75)
for i,v in enumerate(h): ax.text(i,v+4,str(v),ha='center',fontsize=8.5)
ax.set_xticks(range(len(h))); ax.set_xticklabels(lab,fontsize=8.5); ax.set_xlabel('Number of cuisines that use the ingredient'); ax.set_ylabel('Number of ingredients'); ax.set_title(f'{(df_==1).sum()} of {len(ING)} ingredients are used by only one cuisine (removed)',fontsize=11)
ex=[]
for j in np.where(df_==1)[0]:
    c=CU[int(np.argmax(PRES[:,j]))]; ex.append((TF[:,j].max(),ING[j],c))
ex.sort(reverse=True); ax.text(.97,.95,'Examples (cuisine):\n'+'\n'.join(f'{g} ({c})' for _,g,c in ex[:7]),transform=ax.transAxes,ha='right',va='top',fontsize=8.8,color=MUTE)
ax=axs[1]; o=np.argsort(-frac)[:12][::-1]; y=np.arange(12)
for yy,j in zip(y,o):
    c=O if frac[j]>.8 else (Y if frac[j]>.65 else GR); ax.hlines(yy,0,frac[j]*100,color=c,lw=3.2); ax.scatter(frac[j]*100,yy,color=c,s=60,zorder=3)
ax.axvline(80,color=O,ls='--',lw=1.2); ax.axvline(65,color=Y,ls='--',lw=1.2); ax.text(80.6,.2,'0.8 cut-off',color=O,fontsize=8.5); ax.text(65.6,.2,'0.65',color='#a77500',fontsize=8.5)
ax.set_yticks(y); ax.set_yticklabels([ING[j] for j in o],fontsize=10); ax.set_xlabel('% of cuisines using the ingredient'); ax.set_xlim(40,100); ax.set_title('Too common to say anything: removed above the cut-off',fontsize=11)
fig.suptitle('Cleaning the ingredient list before TF-IDF',x=.01,ha='left',fontsize=13,fontweight='bold'); fig.text(.01,.005,'Left: ingredients used by a single cuisine (e.g. wasabi, frog) carry no similarity and are removed — within this cookbook, not in real life. Right: at 0.8 only salt and onion go; at the more balanced 0.65 egg, garlic, sugar, water (and pepper) go too.',fontsize=8,color=MUTE); fig.tight_layout(rect=(0,.03,1,.94)); fig.savefig('figs/c3_ingredient_filters.png'); plt.close()

In [5]:
ANCH={0:(-5.4,2.6),1:(5.4,2.8),2:(-5.2,-4.4),3:(5.6,-4.0),4:(.1,-.6)}; pos={}
for ci,mem in enumerate(big):
    sg=G.subgraph(mem); pl=nx.spring_layout(sg,weight='weight',seed=3,k=2.2/np.sqrt(len(mem)),iterations=500)
    arr=np.array(list(pl.values())); arr=arr-arr.mean(0); arr=arr/np.abs(arr).max()*(.6+.27*len(mem)**.62)
    for (g,_),q in zip(pl.items(),arr): pos[g]=np.array(ANCH[ci])+q
fig,ax=plt.subplots(figsize=(19,13),dpi=200); ax.axis('off')
for ci,mem in enumerate(big):
    pts=np.array([pos[g] for g in mem]); h=pts[ConvexHull(pts).vertices]; cen=h.mean(0); h=cen+(h-cen)*1.1+(h-cen)/np.maximum(np.linalg.norm(h-cen,axis=1,keepdims=True),1e-9)*.3
    ax.fill(h[:,0],h[:,1],color=CC[ci],alpha=.08,zorder=0); ax.plot(np.r_[h[:,0],h[0,0]],np.r_[h[:,1],h[0,1]],color=CC[ci],alpha=.4,lw=1.1,zorder=0); ax.text(cen[0],h[:,1].max()+.03,CN[ci],fontsize=11.5,fontweight='bold',color=CC[ci],ha='center',va='bottom',path_effects=halo)
for u,v,dd in G.edges(data=True):
    if u in pos and v in pos:
        s_=cc[u]==cc[v]; ax.plot([pos[u][0],pos[v][0]],[pos[u][1],pos[v][1]],color='#b9b8b0' if s_ else '#dad9d3',lw=.25+4*(dd['weight']-.3) if s_ else .3,ls='-' if s_ else (0,(3,3)),zorder=1)
for g in G.nodes:
    if g in pos: ax.scatter(*pos[g],s=22+9*G.degree(g),color=CC[cc[g]],edgecolor='white',lw=.8,zorder=3); ax.text(pos[g][0],pos[g][1]+.06,g,fontsize=6.8,ha='center',va='bottom',path_effects=halo,zorder=4)
small=[sorted(s) for s in best if len(s)<8]
ax.text(.01,.01,'Einzelgänger (small groups and cuisines with no link ≥0.3):\n'+'; '.join(', '.join(s) for s in small),transform=ax.transAxes,fontsize=8.2,color=MUTE,va='bottom',wrap=True)
ax.set_title('TF-IDF cuisine graph (cosine ≥ 0.3): the communities Louvain finds',fontsize=14,x=.01)
ax.text(.01,-.01,f'{RULE} Each cuisine is a document, each ingredient a term; edge weight = cosine similarity of TF-IDF vectors, ingredients used by one cuisine or by >80% of cuisines removed. {G.number_of_edges():,} edges; Louvain modularity {X["Q"]:.2f}, stability NMI {X["stab"]:.2f}.',transform=ax.transAxes,fontsize=8.6,color=MUTE,va='top')
fig.savefig('figs/c4_tfidf_network.png',bbox_inches='tight'); plt.close()

In [6]:
E=sorted([(C8[i,j],CU[i],CU[j]) for i in range(N) for j in range(i+1,N)],reverse=True)[:16][::-1]
EXT=set(['South Asian','Caribbean','North American','East Asian','Southeast Asian','North African','South American','Central American','Hausa','Yoruba','Igbo','Efik','Tiv','National Nigerian','Pashtun','Punjabi','Sindhi','Sylheti','Native American','Sundanese','Tibetan','Gibraltarian','Alsatian','Cornish','Manx','Welsh','Scottish','Northern Irish','Hawaiian','Jewish Yemenite','Texas','Southern U.S.','Tex-Mex','Louisiana','Californian','Midwestern U.S.','Southwestern','Italian-American','South Indian'])
fig,ax=plt.subplots(figsize=(10,6),dpi=200)
for y,(w,u,v) in enumerate(E):
    n_=(u in EXT)+(v in EXT); c=[B,Y,O][n_]; ax.hlines(y,.5,w,color=c,lw=3.2); ax.scatter(w,y,color=c,s=65,zorder=3); ax.text(w+.004,y,f'{w:.3f}',fontsize=8.5,va='center',color=MUTE)
ax.set_yticks(range(16)); ax.set_yticklabels([f'{u} – {v}' for _,u,v in E],fontsize=9.5); ax.set_xlim(.5,1.0); ax.set_xlabel('TF-IDF cosine similarity (axis starts at 0.5)')
ax.scatter([],[],color=O,s=60,label='both are ethnic / regional / sub-national'); ax.scatter([],[],color=Y,s=60,label='one is'); ax.scatter([],[],color=B,s=60,label='neither (country-level)'); ax.legend(frameon=False,loc='lower right',fontsize=8.5); nn_=sum((u in EXT) or (v in EXT) for _,u,v in E); ax.set_title('The strongest edges tie a cuisine to its own region, ethnic group or neighbour',fontsize=11.5); fig.text(.01,.005,f'{nn_} of the 16 strongest edges involve a regional, ethnic or sub-national group (e.g. Afghan–Pashtun, Pakistani–Punjabi, Mexican–Tex-Mex).',fontsize=8,color=MUTE)
fig.tight_layout(rect=(0,.03,1,1)); fig.savefig('figs/c7_strongest_edges.png'); plt.close()

In [7]:
Jg=json.load(open('data/world.geojson')); polys=[np.array((f['geometry']['coordinates'][0] if f['geometry']['type']=='Polygon' else None)) for f in Jg['features'] if f['geometry']['type']=='Polygon']
for f in Jg['features']:
    if f['geometry']['type']=='MultiPolygon':
        for p in f['geometry']['coordinates']: polys.append(np.array(p[0]))
fig,axs=plt.subplots(2,3,figsize=(14,6.4),dpi=200); axs=axs.ravel(); sm=[c for c in CU if c not in cc]
for ci in range(6):
    ax=axs[ci]; ax.add_collection(PatchCollection([Polygon(p,closed=True) for p in polys],facecolor='#e6e5e0',edgecolor='#fcfcfb',linewidth=.25)); ax.set_xlim(-130,165); ax.set_ylim(-45,75); ax.set_aspect('equal'); ax.axis('off')
    mem=big[ci] if ci<5 else sm; col_=CC[ci] if ci<5 else MUTE
    for c in mem:
        la,lo_=NA.loc[c,'lat'],NA.loc[c,'lon']
        if not np.isnan(la): ax.scatter(lo_,la,s=26,color=col_,edgecolor='white',lw=.6,alpha=.85,zorder=3)
    ax.set_title(f'{CNl[ci] if ci<5 else "Einzelgänger (small groups, no link ≥0.3)"}  ({len(mem)})',fontsize=9.3,color=col_ if ci<5 else MUTE)
fig.suptitle('Where the TF-IDF communities sit on the map (one dot per cuisine)',x=.01,ha='left',fontsize=12.5,fontweight='bold'); fig.tight_layout(rect=(0,0,1,.94)); fig.savefig('figs/c8_community_maps.png'); plt.close()

In [8]:
u=pd.Series({CU[i]:int(((df_==1)&PRES[i]).sum()) for i in range(N)}).sort_values(ascending=False).head(14)[::-1]; fig,ax=plt.subplots(figsize=(10,5.8),dpi=200)
for y,(c,v) in enumerate(u.items()):
    ax.hlines(y,0,v,color=O,lw=3.2); ax.scatter(v,y,color=O,s=65,zorder=3); jj=np.where((df_==1)&PRES[ix[c]])[0]; jj=jj[np.argsort(-TF[ix[c],jj])][:4]; ax.text(v+.4,y,f'{v}   '+', '.join(ING[j] for j in jj),fontsize=8.3,va='center',color=MUTE)
ax.set_yticks(range(14)); ax.set_yticklabels(u.index,fontsize=10); ax.set_xlim(0,u.max()*2.1); ax.set_xlabel('Ingredients used by this cuisine and no other'); ax.set_title('The most distinctive cuisines: ingredients nobody else uses')
fig.text(.01,.005,'These ingredients are dropped from the similarity graphs but measure uniqueness within this cookbook\n(e.g. truffles count as unique even though they are not in real life).',fontsize=8,color=MUTE); fig.tight_layout(rect=(0,.06,1,1)); fig.savefig('figs/c10_unique_ingredients.png'); plt.close()

## 5. Figures – CuisinesRawCounts graph, twins and strangers, what makes twins (pages 1, 2, 5)

In [9]:
iu=np.triu_indices(N,1); w=RAW[iu]; usn=set(US_ALL); nE=int((w>=1).sum()); st=RAW.sum(1)
TOP=40; o=np.argsort(-w)[:TOP]; E=[(CU[iu[0][k]],CU[iu[1][k]],int(w[k])) for k in o]
H=nx.Graph(); 
for u,v,x in E: H.add_edge(u,v,weight=x)
pos=nx.spring_layout(H,weight='weight',k=1.1,seed=5,iterations=600)
fig=plt.figure(figsize=(16,8),dpi=200); ax=fig.add_axes([.01,.17,.55,.72]); ax.axis('off')
for u,v,x in E: ax.plot(*zip(pos[u],pos[v]),color=O if (u in usn and v in usn) else ('#e3a98f' if (u in usn or v in usn) else '#9bbbe3'),lw=x/14,alpha=.8,solid_capstyle='round',zorder=1)
for n in H.nodes:
    ax.scatter(*pos[n],s=60+st[ix[n]]/14,color=O if n in usn else B,edgecolor='white',lw=1.2,zorder=3)
    ax.text(pos[n][0],pos[n][1]-.055-st[ix[n]]/150000,n,fontsize=9.5,fontweight='bold' if n in usn else 'normal',ha='center',va='top',color=INK,path_effects=halo,zorder=5)
for u,v,x in []:
    m=(pos[u]+pos[v])/2; ax.text(m[0],m[1],str(x),fontsize=9,ha='center',va='center',color=INK,fontweight='bold',bbox=dict(boxstyle='round,pad=0.15',fc='white',ec='none',alpha=.85),zorder=4)
ax.set_title(f'The {TOP} heaviest edges of CuisinesRawCounts',fontsize=13,x=.01)
ax.scatter([],[],s=90,color=O,label='American cuisines (U.S., Southern U.S., Texas, Tex-Mex, Southwestern …)'); ax.scatter([],[],s=90,color=B,label='other cuisines'); h_,l_=ax.get_legend_handles_labels(); fig.legend(h_,l_,frameon=False,loc='lower left',bbox_to_anchor=(.01,.075),ncol=2,fontsize=9.5)
ax2=fig.add_axes([.67,.17,.31,.72]); K=16; ee=E[:K][::-1]
for i,(u,v,x) in enumerate(ee):
    c=O if (u in usn or v in usn) else B; ax2.hlines(i,0,x,color=c,lw=3.4); ax2.scatter(x,i,s=70,color=c,zorder=3); ax2.text(x+1.5,i,str(x),va='center',fontsize=9,color=MUTE)
ax2.set_yticks(range(K)); ax2.set_yticklabels([f'{u} – {v}' for u,v,_ in ee],fontsize=9.5); ax2.set_xlim(0,128); ax2.set_xlabel('ingredients shared (edge weight)')
t10=np.mean([(u in usn or v in usn) for u,v,_ in E[:10]])*100; t25=np.mean([(u in usn or v in usn) for u,v,_ in E[:25]])*100
ax2.set_title(f'{t10:.0f}% of the 10 heaviest edges touch an American cuisine',fontsize=11.5,loc='left')
fig.suptitle('CuisinesRawCounts: counting shared ingredients makes the American cuisines the heaviest nodes',x=.01,ha='left',fontsize=14.5,fontweight='bold')
fig.text(.01,.012,f'Rules: every cuisine is a node ({N}); two cuisines share an undirected edge if they have at least X = 1 ingredient in common; edge weight = raw count of shared ingredients. That gives {nE:,} of {N*(N-1)//2:,} possible edges, median weight 12, maximum 114 (Italian–United States).\nNode size = total weight of its edges (United States 5,089, Indian 4,586, Southern U.S. 4,559). A cuisine with many ingredients shares many ingredients with everyone, which is the bias TF-IDF is meant to remove.',fontsize=8.6,color=MUTE)
fig.savefig('figs/p1_raw_counts_graph.png'); plt.close()

In [10]:
la=NA.reindex(CU)
def hav(a,b):
    p1,l1,p2,l2=map(np.radians,[la.lat[a],la.lon[a],la.lat[b],la.lon[b]]); h=np.sin((p2-p1)/2)**2+np.cos(p1)*np.cos(p2)*np.sin((l2-l1)/2)**2; return 12742*np.arcsin(np.sqrt(h))
keep=(df_>=2)&(frac<=.8); Xk=np.where(TF>0,1+np.log(np.maximum(TF,1)),0)[:,keep]; idf=np.log((1+N)/(1+df_[keep]))+1; W=Xk*idf; W/=np.linalg.norm(W,axis=1,keepdims=True); ik=np.array(ING)[keep]
print('check',np.abs(W@W.T-C8).max())
def shared(a,b,n=4):
    p=W[ix[a]]*W[ix[b]]; return [ik[j] for j in np.argsort(-p)[:n]],p
def distinct(a,b,n=3):
    d_=W[ix[a]]-W[ix[b]]; return [ik[j] for j in np.argsort(-d_)[:n]]
TW=[('Irish','New Zealand'),('Italian','United States'),('Canadian','Russian'),('Gambian','Rwandan'),('Colombian','Italian')]
ST=[('Greek','Hungarian'),('Israeli','Turkish'),('Jamaican','Puerto Rican'),('Algerian','Spanish'),('Filipino','Malaysian')]
fig=plt.figure(figsize=(14,7.6),dpi=200)
for col,(L,ttl,colr) in enumerate([(TW,'Far apart, same kitchen',B),(ST,'Next door, different kitchen',O)]):
    ax=fig.add_axes([.02+col*.505,.09,.47,.78]); ax.set_xlim(0,1); ax.set_ylim(0,5); ax.invert_yaxis(); ax.axis('off')
    fig.text(.02+col*.505,.915,ttl,fontsize=15,fontweight='bold',color=colr)
    for r,(a,b) in enumerate(L):
        km_=hav(a,b); s=C8[ix[a],ix[b]]
        ax.text(0,r+.08,f'{a}  ↔  {b}',fontsize=12.5,fontweight='bold',color=INK,va='top')
        ax.text(1,r+.08,f'{km_:,.0f} km apart',fontsize=10,color=MUTE,va='top',ha='right')
        ax.add_patch(plt.Rectangle((0,r+.42),1,.1,color='#e6e5e0',lw=0)); ax.add_patch(plt.Rectangle((0,r+.42),s,.1,color=colr,lw=0)); ax.text(min(s+.015,.9),r+.47,f'similarity {s:.2f}',fontsize=9,va='center',color=INK)
        if col==0: ax.text(0,r+.62,'shared signature: '+', '.join(shared(a,b)[0]),fontsize=9.5,color=MUTE,va='top')
        else: ax.text(0,r+.62,f'{a}: '+', '.join(distinct(a,b))+f'\n{b}: '+', '.join(distinct(b,a)),fontsize=9.5,color=MUTE,va='top',linespacing=1.5)
fig.text(.02,.012,'Similarity = TF-IDF cosine over ingredients (0 to 1), distance between country centre points. Only cuisines with at least 8 recipes are considered. "Signature" = ingredients that\nmost raise the similarity; for neighbours, the ingredients each cuisine uses far more than the other.',fontsize=8.5,color=MUTE)
fig.suptitle('The surprises: similar cuisines on opposite sides of the planet, and strangers next door',x=.02,ha='left',fontsize=15,fontweight='bold',y=.985)
fig.savefig('figs/p2_twins_and_strangers.png'); plt.close()
fig,axs=plt.subplots(1,3,figsize=(15,5.2),dpi=200)
for ax,(a,b),c in zip(axs,[('Italian','United States'),('Irish','New Zealand'),('Canadian','Russian')],[B,A,V]):
    _,p=shared(a,b); o=np.argsort(-p)[:9][::-1]; sh=p[o]/(W[ix[a]]@W[ix[b]])*100
    ax.hlines(range(9),0,sh,color=c,lw=3.2); ax.scatter(sh,range(9),color=c,s=60,zorder=3); ax.set_yticks(range(9)); ax.set_yticklabels(ik[o],fontsize=10)
    ax.set_xlim(0,sh.max()*1.25); ax.set_xlabel('% of the similarity score'); ax.set_title(f'{a} ↔ {b}  ({C8[ix[a],ix[b]]:.2f})',fontsize=12)
    for y_,v in zip(range(9),sh): ax.text(v+sh.max()*.02,y_,f'{v:.0f}%',va='center',fontsize=8.5,color=MUTE)
fig.suptitle('What makes the twins twins: the ingredients that carry their similarity',x=.01,ha='left',fontsize=14,fontweight='bold')
fig.text(.01,.01,'Each bar is one ingredient’s share of the cosine similarity between the two cuisines (the nine largest of all shared ingredients).',fontsize=8.5,color=MUTE)
fig.tight_layout(rect=(0,.04,1,.94)); fig.savefig('figs/p5_what_makes_twins.png'); plt.close()
print('done')

check 1.0000000000000009


done


## 6. Figure – small ethnic and indigenous cuisines (page 9)

In [11]:
from networkx.algorithms.community import louvain_communities,modularity
SMALL=['Pashtun','Sindhi','Punjabi','Sylheti','Hausa','Yoruba','Igbo','Efik','Tiv','National Nigerian','Native American','Hawaiian','Tibetan','Sundanese','Gibraltarian','Jewish Yemenite','Alsatian','Cornish','Manx','Welsh','Scottish','Northern Irish','Sabbath']
SMALL=[s for s in SMALL if s in ix]
GRP=[('West Africa (Nigeria)',A,['National Nigerian','Yoruba','Hausa','Igbo','Efik','Tiv']),('South Asia & Himalaya',V,['Punjabi','Sindhi','Pashtun','Sylheti','Tibetan']),('Jewish communities',Y,['Jewish Yemenite','Sabbath']),('British Isles & Europe',B,['Scottish','Welsh','Cornish','Northern Irish','Manx','Gibraltarian','Alsatian']),('Indigenous & island',O,['Native American','Hawaiian','Sundanese'])]
rows=[]
for g,col,L in GRP:
    for s_ in L:
        if s_ not in ix: continue
        o=[j for j in np.argsort(-C8[ix[s_]]) if j!=ix[s_] and CU[j] not in SMALL][:3]; rows.append((g,col,s_,[(CU[j],C8[ix[s_],j]) for j in o]))
fig,ax=plt.subplots(figsize=(12.5,8.6),dpi=200); y=0; yt=[]; yl=[]; ygrp=[]
for g,col,L in GRP:
    y0=y
    for gg,cc_,s_,nb in [r for r in rows if r[0]==g]:
        v=nb[0][1]; few=nrec[s_]<5
        ax.hlines(y,0,v,color=col,lw=3,alpha=.35 if few else 1)
        ax.scatter(v,y,s=75,color='white' if few else col,edgecolor=col,lw=2,zorder=3)
        ax.text(v+.012,y,'  ·  '.join(f'{c} {w:.2f}' for c,w in nb),fontsize=8.8,va='center',color=INK)
        yt.append(y); yl.append(f'{s_}  ({nrec[s_]})'); y+=1
    ygrp.append((g,col,(y0+y-1)/2,y0-.5,y-.5)); y+=.9
ax.set_yticks(yt); ax.set_yticklabels(yl,fontsize=9.5); ax.invert_yaxis(); ax.set_xlim(0,1.8); ax.set_xticks([0,.2,.4,.6,.8,1.0])
for g,col,ym,y0,y1 in ygrp: ax.text(1.79,ym,g,fontsize=11,fontweight='bold',color=col,ha='right',va='center')
ax.set_xlabel('TF-IDF cosine similarity to the closest larger cuisine'); fig.text(.01,.965,'Small ethnic and indigenous cuisines: who they resemble most',fontsize=13,fontweight='bold')
h1=ax.scatter([],[],s=60,color=GR,label='5 or more recipes'); h2=ax.scatter([],[],s=60,color='white',edgecolor=GR,lw=2,label='fewer than 5 recipes (treat with caution)'); fig.legend(handles=[h1,h2],frameon=False,loc='upper right',bbox_to_anchor=(.985,.975),fontsize=9)
fig.text(.01,.005,'Recipes in brackets. Labels list the three most similar cuisines outside this set of small groups, with cosine similarity. Regional and ethnic groups resemble their own country or\nregion (Hausa/Yoruba → Nigerian, Pashtun → Afghan, Cornish/Welsh → English); the 2-recipe groups (Hawaiian, Manx, Sundanese) are too small to read much into.',fontsize=8.2,color=MUTE)
fig.tight_layout(rect=(0,.04,1,.95)); fig.savefig('figs/c5_small_and_indigenous_groups.png'); plt.close()
for g,col,s_,nb in rows: print(s_,nrec[s_],[(c,round(w,2)) for c,w in nb])

National Nigerian 38 [('Nigerian', np.float64(0.54)), ('Liberian', np.float64(0.48)), ('Ugandan', np.float64(0.4))]
Yoruba 24 [('Nigerian', np.float64(0.7)), ('Cameroonian', np.float64(0.54)), ('Liberian', np.float64(0.49))]
Hausa 33 [('Nigerian', np.float64(0.59)), ('Liberian', np.float64(0.38)), ('Cameroonian', np.float64(0.38))]
Igbo 13 [('Nigerian', np.float64(0.65)), ('Cameroonian', np.float64(0.54)), ('Liberian', np.float64(0.3))]
Efik 6 [('Nigerian', np.float64(0.55)), ('Cameroonian', np.float64(0.33)), ('Liberian', np.float64(0.28))]
Tiv 7 [('Nigerian', np.float64(0.4)), ('Cameroonian', np.float64(0.35)), ('Liberian', np.float64(0.28))]
Punjabi 6 [('Pakistani', np.float64(0.88)), ('Afghan', np.float64(0.68)), ('Indian', np.float64(0.55))]
Sindhi 16 [('Indian', np.float64(0.63)), ('South Indian', np.float64(0.61)), ('Pakistani', np.float64(0.54))]
Pashtun 2 [('Afghan', np.float64(0.97)), ('Pakistani', np.float64(0.58)), ('Ethiopian', np.float64(0.4))]
Sylheti 8 [('South Asian', 

## 7. Figures – recipe types and recipe-based cuisine groups (page 10), religion and far-apart cuisines (page 11)

Recipes are clustered into 9 types by ingredient content (TF-IDF, SVD, k-means); each cuisine is described by its mix of types and grouped with Louvain on a nearest-neighbour graph.

In [12]:
from sklearn.feature_extraction.text import TfidfTransformer
from sklearn.decomposition import TruncatedSVD
from sklearn.cluster import KMeans
dd=base.d; dd=dd.assign(rid=dd.cuisine+'||'+dd.recipe.astype(str))
R_=dd.groupby('rid'); rids=sorted(dd.rid.unique()); rix={r:i for i,r in enumerate(rids)}
M=np.zeros((len(rids),len(ING)),dtype=np.float32)
for r,g in zip(dd.rid,dd.ingredient): M[rix[r],jx[g]]=1
rc=np.array([r.split('||')[0] for r in rids]); print(M.shape)
dfr=M.sum(0); keep=(dfr>=6)&(dfr/len(rids)<=.25); print('ingredients kept',keep.sum())
T=TfidfTransformer(sublinear_tf=True).fit_transform(M[:,keep]); 
Zs=TruncatedSVD(40,random_state=0).fit_transform(T); Zs/=np.maximum(np.linalg.norm(Zs,axis=1,keepdims=True),1e-9)
K=9; km=KMeans(K,n_init=20,random_state=1).fit(Zs); lab_r=km.labels_
ingk=np.array(ING)[keep]; Tn=np.asarray(T.todense())
for k in range(K):
    m=lab_r==k; sc=Tn[m].mean(0)-Tn.mean(0); print(k,m.sum(),[ingk[j] for j in np.argsort(-sc)[:9]], pd.Series(rc[m]).value_counts().head(4).to_dict())
pickle.dump(dict(rids=rids,rc=rc,lab=lab_r),open('work/rtypes.pkl','wb'))
from networkx.algorithms.community import louvain_communities,modularity
cs=[c for c in CU if (rc==c).sum()>=6]; print(len(cs),'cuisines >=6 recipes')
PR=np.array([[np.mean(lab_r[rc==c]==k) for k in range(K)] for c in cs])
Pn=PR/np.linalg.norm(PR,axis=1,keepdims=True); SM=Pn@Pn.T; np.fill_diagonal(SM,0)
Gc=nx.Graph(); Gc.add_nodes_from(range(len(cs)))
for i in range(len(cs)):
    for j in np.argsort(-SM[i])[:4]: Gc.add_edge(i,int(j),weight=float(SM[i,j]))
bq,bl=-1,None
for sd in range(80):
    l=louvain_communities(Gc,weight='weight',seed=sd); q=modularity(Gc,l,weight='weight')
    if q>bq: bq,bl=q,l
bl=sorted(bl,key=len,reverse=True); print('Q',round(bq,2))
for k,g in enumerate(bl): print(k,len(g),[cs[i] for i in sorted(g)])
pickle.dump(dict(cs=cs,PR=PR,bl=[sorted(g) for g in bl],bq=bq),open('work/rcomm.pkl','wb'))

(2096, 563)
ingredients kept 299
0 325 [np.str_('soy sauce'), np.str_('lime'), np.str_('vegetable oil'), np.str_('chicken'), np.str_('pork'), np.str_('tortilla'), np.str_('green onion'), np.str_('sour cream'), np.str_('cilantro')] {'Southwestern': 27, 'United States': 24, 'Chinese': 22, 'Japanese': 22}
1 181 [np.str_('water'), np.str_('sugar'), np.str_('dough'), np.str_('rice flour'), np.str_('millet'), np.str_('food coloring'), np.str_('cassava'), np.str_('semolina'), np.str_('yeast')] {'Indian': 15, 'Hausa': 11, 'Nigerian': 11, 'Japanese': 9}
2 204 [np.str_('olive oil'), np.str_('parmesan cheese'), np.str_('basil'), np.str_('pasta'), np.str_('oregano'), np.str_('mozzarella cheese'), np.str_('tomato'), np.str_('rosemary'), np.str_('tomato paste')] {'Italian': 66, 'United States': 20, 'French': 15, 'Greek': 10}
3 216 [np.str_('sugar'), np.str_('cinnamon'), np.str_('coconut'), np.str_('cardamom'), np.str_('almond'), np.str_('raisin'), np.str_('vanilla'), np.str_('clove'), np.str_('pista

Q 0.72
0 22 ['Alsatian', 'British', 'Bulgarian', 'Colombian', 'Cornish', 'Danish', 'Dutch', 'English', 'French', 'Irish', 'Louisiana', 'Midwestern U.S.', 'New Zealand', 'North American', 'Polish', 'Romanian', 'Russian', 'Scottish', 'South African', 'Southern U.S.', 'Swedish', 'United States']
1 17 ['Californian', 'Chinese', 'Filipino', 'Ghanaian', 'Indonesian', 'Iranian', 'Jamaican', 'Japanese', 'Korean', 'Malawian', 'Mexican', 'Peruvian', 'Southwestern', 'Tex-Mex', 'Thai', 'Ugandan', 'Vietnamese']
2 13 ['Algerian', 'Bengali', 'Caribbean', 'Indian', 'Kenyan', 'Malaysian', 'Pakistani', 'Punjabi', 'Sindhi', 'South Asian', 'South Indian', 'Sylheti', 'Turkish']
3 10 ['Australian', 'Austrian', 'Canadian', 'Finnish', 'German', 'Hungarian', 'Jewish Yemenite', 'Trinidadian and Tobagonian', 'Venezuelan', 'Welsh']
4 10 ['Brazilian', 'Cameroonian', 'Efik', 'Hausa', 'Igbo', 'Liberian', 'National Nigerian', 'Nigerian', 'Tiv', 'Yoruba']
5 10 ['Egyptian', 'Ethiopian', 'Gambian', 'Israeli', 'Libyan', 

In [13]:
from sklearn.metrics import normalized_mutual_info_score as nmi
Rr=pickle.load(open('work/rcomm.pkl','rb')); cs,PR,bl,bq=Rr['cs'],Rr['PR'],Rr['bl'],Rr['bq']
TN=['Soy, lime & tortilla dishes','Dough, rice flour & porridges','Italian pasta & herbs','Spiced sweets','Baking','Paprika–cumin stews','Palm-oil fish stews','Curries','Butter, bacon & potato']
PC=np.array([PR[g].mean(0) for g in bl]); lab=np.zeros(len(cs),int)
for k,g in enumerate(bl):
    for i in g: lab[i]=k
Pn=PR/np.linalg.norm(PR,axis=1,keepdims=True); SM=Pn@Pn.T; np.fill_diagonal(SM,0)
Gc=nx.Graph()
for i in range(len(cs)):
    Gc.add_node(i)
    for j in np.argsort(-SM[i])[:4]: Gc.add_edge(i,int(j),weight=float(SM[i,j]))
pos=nx.spring_layout(Gc,weight='weight',k=.45,seed=11,iterations=600)
SC=CC+['#7a7a76','#c23d8e']
LET='ABCDEFG'
names=[' + '.join(TN[k] for k in np.argsort(-PC[g])[:2]) for g in range(len(bl))]
fig=plt.figure(figsize=(16,8.2),dpi=200); ax=fig.add_axes([.01,.07,.58,.82]); ax.axis('off')
for u,v in Gc.edges(): ax.plot(*zip(pos[u],pos[v]),color='#d4d3ce' if lab[u]!=lab[v] else SC[lab[u]],lw=.5,alpha=.55,zorder=1)
for i,c in enumerate(cs): ax.scatter(*pos[i],s=30+8*np.sqrt(nrec[c]),color=SC[lab[i]],edgecolor='white',lw=.7,zorder=3)
for i,c in enumerate(cs): ax.text(pos[i][0],pos[i][1]-.022,c,fontsize=6.6,ha='center',va='top',color=INK,path_effects=halo,zorder=5)
for g in range(len(bl)):
    cx=np.mean([pos[i][0] for i in bl[g]]); cy=np.mean([pos[i][1] for i in bl[g]]); ax.text(cx,cy,LET[g],fontsize=26,fontweight='bold',color=SC[g],alpha=.28,ha='center',va='center',zorder=0)
ax.set_title('Cuisines grouped by the kinds of recipes they cook, not by where they are',fontsize=13.5,x=.01,y=1.02)
ax2=fig.add_axes([.645,.13,.29,.66]); im=ax2.imshow(PC*100,cmap='Blues',aspect='auto',vmin=0,vmax=70)
for i in range(PC.shape[0]):
    for j in range(PC.shape[1]):
        if PC[i,j]>=.08: ax2.text(j,i,f'{PC[i,j]*100:.0f}',ha='center',va='center',fontsize=8.5,color='white' if PC[i,j]>.35 else INK)
ax2.set_xticks(range(len(TN))); ax2.set_xticklabels(TN,rotation=45,ha='left',fontsize=8.5); ax2.xaxis.tick_top()
ax2.set_yticks(range(len(bl))); ax2.set_yticklabels([f'{LET[g]}  ({len(bl[g])})' for g in range(len(bl))],fontsize=11,fontweight='bold')
for t,g in zip(ax2.get_yticklabels(),range(len(bl))): t.set_color(SC[g])
for sp in ax2.spines.values(): sp.set_visible(False)
fig.text(.645,.075,'Percent of each group’s recipes that fall in each recipe type.\nGroup size in brackets (cuisines with ≥ 6 recipes).',fontsize=8.5,color=MUTE)
ct=NA.reindex(cs).continent; okk=ct.notna().values; n_=nmi(ct[okk],lab[okk])
rng=np.random.default_rng(0); null=np.percentile([nmi(ct[okk],rng.permutation(lab[okk])) for _ in range(500)],95)
fig.text(.01,.012,f'Recipes (2,096) were clustered into 9 types by ingredient content; each cuisine is described by its mix of types, and cuisines are grouped by Louvain on their 4 nearest neighbours (modularity {bq:.2f}).\nAfterwards, agreement with continent is NMI {n_:.2f} (95th percentile of shuffles {null:.2f}): real but partial. Groups mix places, e.g. Jamaica with Japan and Korea, Texas with Ethiopia, Native American with Italian.',fontsize=8.5,color=MUTE)
fig.savefig('figs/p10_recipe_groups.png'); plt.close()
for g in range(len(bl)): print(LET[g],names[g],[cs[i] for i in bl[g]][:6])

A Butter, bacon & potato + Baking ['Alsatian', 'British', 'Bulgarian', 'Colombian', 'Cornish', 'Danish']
B Soy, lime & tortilla dishes + Spiced sweets ['Californian', 'Chinese', 'Filipino', 'Ghanaian', 'Indonesian', 'Iranian']
C Curries + Spiced sweets ['Algerian', 'Bengali', 'Caribbean', 'Indian', 'Kenyan', 'Malaysian']
D Baking + Butter, bacon & potato ['Australian', 'Austrian', 'Canadian', 'Finnish', 'German', 'Hungarian']
E Palm-oil fish stews + Soy, lime & tortilla dishes ['Brazilian', 'Cameroonian', 'Efik', 'Hausa', 'Igbo', 'Liberian']
F Paprika–cumin stews + Spiced sweets ['Egyptian', 'Ethiopian', 'Gambian', 'Israeli', 'Libyan', 'Puerto Rican']
G Italian pasta & herbs + Butter, bacon & potato ['Albanian', 'Cuban', 'Gibraltarian', 'Greek', 'Italian', 'Italian-American']


Religion: Pew composition per country, applied to the parent country of sub-national cuisines. Far pairs = 4,000 km or more apart, no shared official language, no colonial link.

In [14]:
RL=pd.read_csv('data/religion_by_cuisine.csv',index_col=0)
rcols=['christian','muslim','hindu','buddhist','folk','jewish','unaffiliated']
NA2=NA.copy(); ok=[c for c in CU if c in RL.index and pd.notna(RL.loc[c,'christian']) and pd.notna(NA2.loc[c,'lat']) and pd.notna(NA2.loc[c,'country_iso'])]
n=len(ok); I=[ix[c] for c in ok]; print(n,'cuisines with religion, location and country')
sh=RL.loc[ok,rcols].fillna(0).values.astype(float); sh=sh/sh.sum(1,keepdims=True)
lat=np.radians(NA2.loc[ok,'lat'].values); lon=np.radians(NA2.loc[ok,'lon'].values)
iso=NA2.loc[ok,'country_iso'].values; col=NA2.loc[ok,'coloniser_group'].values; langs=[set(str(x).split(';')) if pd.notna(x) else set() for x in NA2.loc[ok,'official_languages']]
plur=RL.loc[ok,'plurality_religion'].values
iu=np.triu_indices(n,1); a,b=iu
d=12742*np.arcsin(np.sqrt(np.sin((lat[b]-lat[a])/2)**2+np.cos(lat[a])*np.cos(lat[b])*np.sin((lon[b]-lon[a])/2)**2))
keep=iso[a]!=iso[b]; a,b,d=a[keep],b[keep],d[keep]
y=C8[np.ix_(I,I)][a,b]

keepi=(df_>=2)&(frac<=.8); Xk=np.where(TF>0,1+np.log(np.maximum(TF,1)),0)[:,keepi]; idf=np.log((1+N)/(1+df_[keepi]))+1; W=Xk*idf; W/=np.linalg.norm(W,axis=1,keepdims=True); ik=np.array(ING)[keepi]
def sig(p,q,n=4):
    v=W[ix[p]]*W[ix[q]]; return ', '.join(ik[j] for j in np.argsort(-v)[:n])
lang=np.array([len(langs[i]&langs[j])>0 for i,j in zip(a,b)]); colo=((col[a]==col[b])&(col[a]!='none/other'))
far=(d>=4000)&(~lang)&(~colo); pl=np.array(plur)
cn=np.array([NA2.loc[c,'kind']=='country' for c in ok])
def cat_means(p):
    out={}
    for r1,r2_ in CATS:
        m=far&(((p[a]==r1)&(p[b]==r2_))|((p[a]==r2_)&(p[b]==r1))); out[(r1,r2_)]=y[m].mean() if m.sum() else np.nan
    return out
CATS=[('muslim','muslim'),('christian','christian'),('christian','muslim'),('christian','unaffiliated'),('muslim','unaffiliated'),('buddhist','christian')]
obs=cat_means(pl); rng=np.random.default_rng(0); nulls=[cat_means(rng.permutation(pl)) for _ in range(500)]
cnt={k:int((far&(((pl[a]==k[0])&(pl[b]==k[1]))|((pl[a]==k[1])&(pl[b]==k[0])))).sum()) for k in CATS}
allfar=y[far].mean(); print('overall far mean',round(allfar,3))
LB={'muslim':'Muslim','christian':'Christian','unaffiliated':'Unaffiliated','buddhist':'Buddhist'}
fig=plt.figure(figsize=(17,7.4),dpi=200)
ax=fig.add_axes([.155,.25,.2,.55]); order=sorted(CATS,key=lambda k:-obs[k])
for i,k in enumerate(order):
    nl=np.array([n_[k] for n_ in nulls]); lo,hi=np.nanpercentile(nl,[2.5,97.5]); ax.add_patch(plt.Rectangle((lo,i-.3),hi-lo,.6,color='#e6e5e0',lw=0,zorder=1))
    ax.hlines(i,0,obs[k],color=B,lw=3.2,zorder=2); ax.scatter(obs[k],i,s=90,color=B,zorder=3); ax.text(obs[k]+.01,i,f'{obs[k]:.2f}',va='center',fontsize=9)
ax.set_yticks(range(len(order))); ax.set_yticklabels([f'{LB[k[0]]} – {LB[k[1]]}\n{cnt[k]:,} pairs' for k in order],fontsize=9); ax.set_ylim(len(order)-.4,-.7); ax.set_xlim(0,.3); ax.set_xlabel('mean TF-IDF similarity')
ax.axvline(allfar,color=GR,ls=':',lw=1.2); ax.text(allfar+.004,-.66,f'all far pairs: {allfar:.2f}',fontsize=8,color=MUTE,ha='left',va='top')
ax.set_title('Far apart, same religion: only slightly closer',loc='left',fontsize=11.5)
fig.text(.155,.125,f'Grey band = range expected if religion labels were shuffled (95%).\nPairs 4,000 km+ apart, no shared language or colonial link.\nAll same-religion pairs together: {y[far&(pl[a]==pl[b])].mean():.2f} vs {y[far&(pl[a]!=pl[b])].mean():.2f} for different religions (shuffle test p = 0.02).',fontsize=8,color=MUTE)
ax=fig.add_axes([.5,.25,.2,.55]); PV=TF/np.maximum(nrec.reindex(CU).values[:,None],1); nr=np.array([nrec[c] for c in ok]); msel=cn&(nr>=6)
mu=np.where((pl=='muslim')&msel)[0]; ch=np.where((pl=='christian')&msel)[0]; P_=PV[np.ix_(I,range(len(ING)))]
IL=['bacon','ham','pork','sausage','lard','turmeric','cardamom','cumin','almond','chickpea']
sy=np.arange(len(IL)); 
for i,g in enumerate(IL):
    pc=(P_[ch,jx[g]]>0).mean()*100; pm=(P_[mu,jx[g]]>0).mean()*100
    ax.hlines(i,pc,pm,color='#c9c8c3',lw=3); ax.scatter(pc,i,s=85,color=B,zorder=3); ax.scatter(pm,i,s=85,color=A,zorder=3)
ax.axhline(4.5,color='#d4d3ce',lw=.8); ax.set_yticks(sy); ax.set_yticklabels(IL,fontsize=9.5); ax.set_ylim(len(IL)-.4,-.6); ax.set_xlim(-3,100); ax.set_xlabel('% of cuisines that use it')
ax.scatter([],[],s=70,color=B,label=f'Christian-majority ({len(ch)})'); ax.scatter([],[],s=70,color=A,label=f'Muslim-majority ({len(mu)})'); ax.legend(frameon=False,loc='upper right',bbox_to_anchor=(1.04,1.02),fontsize=8.5)
ax.set_title('What travels with religion: pork and spices',loc='left',fontsize=11.5)
pk=[jx[g] for g in ['pork','bacon','ham','sausage','lard']]; usep=(P_[:,pk]>0).any(1); cont=NA2.loc[ok,'continent'].values
afr=lambda r:(pl==r)&cn&(cont=='Africa')
fig.text(.5,.1,f'Any pork product: {usep[(pl=="christian")&cn].mean()*100:.0f}% of Christian-majority vs {usep[(pl=="muslim")&cn].mean()*100:.0f}% of Muslim-majority cuisines.\nWithin Africa alone: {usep[afr("christian")].mean()*100:.0f}% (n={afr("christian").sum()}) vs {usep[afr("muslim")].mean()*100:.0f}% (n={afr("muslim").sum()}).',fontsize=8.5,color=MUTE)
ax=fig.add_axes([.775,.12,.215,.7]); ax.axis('off'); ax.set_xlim(0,1); ax.set_ylim(7.2,0)
rowsC=[('Same religion, far apart',[('Egyptian','Senegalese'),('Libyan','Pakistani'),('Bulgarian','Russian')]),('Different religion, same spice cabinet',[('Kenyan','Senegalese'),('Ethiopian','Pakistani'),('Ethiopian','Indian')])]
yy=0
for ttl,L in rowsC:
    ax.text(0,yy,ttl,fontsize=10.5,fontweight='bold',color=INK); yy+=.75
    for p,q in L:
        ia,ib=ok.index(p),ok.index(q); k_=np.where(((a==ia)&(b==ib))|((a==ib)&(b==ia)))[0][0]
        ax.text(0,yy,f'{p} – {q}',fontsize=10,fontweight='bold',color=INK); ax.text(1,yy,f'{LB.get(pl[ia],pl[ia].title())} / {LB.get(pl[ib],pl[ib].title())} · {d[k_]:,.0f} km · {y[k_]:.2f}',fontsize=7.8,color=MUTE,ha='right')
        ax.text(0,yy+.42,sig(p,q),fontsize=8.8,color=MUTE); yy+=1.05
    yy+=.45
fig.suptitle('Do far-apart cuisines that share a religion cook alike? Only slightly, and the clearest link is what religion rules out',x=.01,ha='left',fontsize=14.5,fontweight='bold',y=.975)
fig.text(.01,.012,f'{n} cuisines with Pew religion shares; religion = plurality. Ingredient panel uses country cuisines with at least 6 recipes (Muslim-majority n is small). Similarity = TF-IDF cosine; signature = ingredients adding most to the similarity.',fontsize=8.3,color=MUTE)
fig.savefig('figs/p11_religion.png'); plt.close()
for k in order: print(k,cnt[k],round(obs[k],3),np.round(np.nanpercentile([n_[k] for n_ in nulls],[2.5,97.5]),3))

129 cuisines with religion, location and country


overall far mean 0.166


('muslim', 'muslim') 111 0.201 [0.124 0.207]
('christian', 'christian') 1911 0.18 [0.152 0.181]
('christian', 'muslim') 1187 0.164 [0.149 0.182]
('christian', 'unaffiliated') 534 0.162 [0.124 0.204]
('muslim', 'unaffiliated') 137 0.136 [0.116 0.21 ]
('buddhist', 'christian') 225 0.13 [0.093 0.227]


## 8. Check against the supplied TF-IDF network (`FiltredIngredientcuisines65.gexf`)

In [15]:
Gu=nx.read_gexf('data/FiltredIngredientcuisines65.gexf')
common=[c for c in CU if c in Gu]
print('GEXF:',Gu.number_of_nodes(),'nodes,',Gu.number_of_edges(),'edges; cuisines in both:',len(common))
C65=X['C65']; ours={}
for i,u in enumerate(CU):
    for j in range(i+1,len(CU)):
        if C65[i,j]>=.3 and u in Gu and CU[j] in Gu: ours[frozenset((u,CU[j]))]=C65[i,j]
theirs={frozenset((u,v)):float(dd.get('weight',1)) for u,v,dd in Gu.edges(data=True) if u in set(common) and v in set(common)}
both=set(ours)&set(theirs); print('edges at cosine >= 0.3 (0.65 filter): ours',len(ours),'| GEXF',len(theirs),'| shared',len(both),'| Jaccard',round(len(both)/len(set(ours)|set(theirs)),2))
print('weight correlation on shared edges:',round(np.corrcoef([ours[e] for e in both],[theirs[e] for e in both])[0,1],3))

GEXF: 152 nodes, 1169 edges; cuisines in both: 144
edges at cosine >= 0.3 (0.65 filter): ours 1066 | GEXF 1010 | shared 1002 | Jaccard 0.93
weight correlation on shared edges: 0.993


## 9. Tables

In [16]:
iu=np.triu_indices(N,1)
pd.DataFrame([(CU[i],CU[j],int(RAW[i,j])) for i,j in zip(*iu) if RAW[i,j]>=1],columns=['cuisine_a','cuisine_b','shared_ingredients']).to_csv('tables/CuisinesRawCounts_edges.csv',index=False)
pd.DataFrame([(CU[i],CU[j],round(float(C8[i,j]),4)) for i,j in zip(*iu) if C8[i,j]>=.3],columns=['cuisine_a','cuisine_b','tfidf_cosine']).to_csv('tables/TFIDF_edges_cos0.3.csv',index=False)
lab=[-1]*N
for k,g in enumerate(best):
    for m in g: lab[CU.index(m) if isinstance(m,str) else m]=k
pd.DataFrame({'cuisine':CU,'tfidf_community':lab}).to_csv('tables/cuisine_communities.csv',index=False)
pd.DataFrame({'ingredient':ING,'cuisines_using':df_,'share_of_cuisines':np.round(df_/N,3),'single_cuisine':df_==1}).sort_values('cuisines_using',ascending=False).to_csv('tables/ingredient_filter_report.csv',index=False)
pd.DataFrame([(chr(65+k),cs[i]) for k,g in enumerate(bl) for i in g],columns=['recipe_group','cuisine']).to_csv('tables/recipe_based_cuisine_groups.csv',index=False)
print(os.listdir('tables'))

['recipe_based_cuisine_groups.csv', 'cuisine_communities.csv', 'ingredient_filter_report.csv', 'CuisinesRawCounts_edges.csv', 'TFIDF_edges_cos0.3.csv']
